In [ ]:
!pip install -q -U google-genai sentence-transformers chromadb langchain-text-splitters pypdf

In [ ]:
!pip install chromadb

In [ ]:
import os
from google.colab import userdata, files
from google import genai
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pypdf import PdfReader

In [ ]:
api_key = os.environ.get("GOOGLE_API_KEY")

if not api_key:
    try:
        api_key = userdata.get("Gemini_API_Key_2")
    except Exception:
        pass

if not api_key:
    raise ValueError(
        "Gemini API key not found. "
        "Please add Gemini_API_Key_2 to Colab Secrets."
    )

os.environ["GOOGLE_API_KEY"] = api_key

client = genai.Client(api_key=api_key)

print("Gemini API setup complete!")

Gemini API setup complete!


In [ ]:
print("Please upload one or more files:")
uploaded = files.upload()

pdf_texts = []
for filename in uploaded.keys():
  if filename.endswith(".pdf"):
    reader = PdfReader(filename)
    text =""
    for page_num, page in enumerate(reader.pages):
      page_text = page.extract_text()
      if page_text:
        text += f"\n--- Page {page_num +1} ---\n" + page_text
        pdf_texts.append(text)
        print(f" Loaded '{filename}' ({len(reader.pages)} paegs).")
if not pdf_texts:
  raise ValueError("No valid PDF files uploaded. Please rerun and upload a.pdf")
full_pdf_content = "\n\n".join(pdf_texts)



Please upload one or more files:


Saving GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf to GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).
 Loaded 'GenAI_10_Day_30_Hour_Workshop_Curriculum(1).pdf' (11 paegs).


In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 100
    )
chunks = text_splitter.split_text(full_pdf_content)
print(f" Extracted and split document into{len(chunks)} text chunks." )


 Extracted and split document into130 text chunks.


In [ ]:
print("Loading embedding model and building vector index...")
embedder = SentenceTransformer("all-MiniLM-L6-v2")
chroma_client = chromadb.Client()

# Reset collection for clean execution
try:
  chroma_client.delete_collection(name="pdf_rag_collection")
except Exception:
  pass
collection = chroma_client.create_collection(name="pdf_rag_collection")

# Embed chunks in batches
chunk_embeddings = embedder.encode(chunks).tolist()
chunk_ids = [f"doc_chunk_{i}" for i in range(len(chunks))]

collection.add(
    documents = chunks,
    embeddings = chunk_embeddings,
    ids = chunk_ids
    )
print(" PDF Vector Indexing Complete:\n")



Loading embedding model and building vector index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 PDF Vector Indexing Complete:



In [ ]:
def retrieve_pdf_context(query: str, top_k: int = 3) -> list[str]:
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )
    return results["documents"][0]

def ask_pdf(query: str):
    context_passages = retrieve_pdf_context(query, top_k=3)
    context_str = "\n".join(f"- {p}" for p in context_passages)

    prompt = f"""Answer the user's question using only the PDF; if the answer is not found, say "I could not find this information in the uploaded PDF." Keep the answer simple and do not invent information.

PDF Context:
{context_str}

Question: {query}
Answer:"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt
    )

    return response.text, context_passages

# The loop must be un-indented so it runs outside ask_pdf
while True:
    user_query = input("\nAsk a question about your PDF: ")
    if user_query.lower() in ["exit", "quit", "q"]:
        print("Exiting PDF Chatbot. Goodbye!")
        break
    if not user_query.strip():
        continue

    answer, context = ask_pdf(user_query)

    print("\n--- RETRIEVED PDF SNIPPETS ---")
    for i, snippet in enumerate(context, 1):
        print(f"[{i}] {snippet[:150]}...")

    print("\n--- GEMINI RESPONSE ---")
    print(answer)
    print("_" * 60)


--- RETRIEVED PDF SNIPPETS ---
[1] • Chunk size and overlap
 • Embedding generation
 • Vector store creation
 • Retrieval strategies
 • Grounded response generation
 • Source-aware answ...
[2] 3 HOURS
TOPICS COVERED
 • What is Multimodal AI?
 • Text + image understanding
 • Vision models
 • Image understanding workflows
 • OCR — Optical Char...
[3] 3 HOURS
TOPICS COVERED
 • What is Multimodal AI?
 • Text + image understanding
 • Vision models
 • Image understanding workflows
 • OCR — Optical Char...

--- GEMINI RESPONSE ---
Based on the provided text, the PDF outlines topics covered in training sessions (divided into 3-hour sections). It covers:

* **Building and evaluating a PDF chatbot:** Chunk size and overlap, embedding generation, vector stores, retrieval strategies, grounded/source-aware responses, and improving retrieval quality.
* **Multimodal AI and Vision:** Text + image understanding, vision models, image workflows, and OCR (Optical Character Recognition).
* **GenAI Techniq

In [9]:
!pip uninstall -y mcp fastmcp langchain-mcp-adapters langgraph langchain-google-genai
!pip install -q "mcp>=1.0.0,<2.0.0" mcp-types langgraph langchain-google-genai nest_asyncio

Found existing installation: langgraph 1.2.11
Uninstalling langgraph-1.2.11:
  Successfully uninstalled langgraph-1.2.11
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 787.2 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.6/234.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.9/248.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.6/81.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.7/571.7 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.0/260.0 kB 20.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency c

In [10]:
import os
import csv
import asyncio
import nest_asyncio
from google.colab import userdata

# LangChain & LangGraph Imports
from langchain_core.tools import StructuredTool
from langgraph.prebuilt import create_react_agent
from langchain_google_genai import ChatGoogleGenerativeAI

nest_asyncio.apply()

# 1. Setup Gemini API Key from Colab Secrets
os.environ["GOOGLE_API_KEY"] = userdata.get("Gemini_API_Key_2")

# 2. Initialize Gemini LLM (Use valid model name)
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", temperature=0)

# -------------------------------------------------------------
# 3. Local Real-World Tools (MCP Primitive Logic)
# -------------------------------------------------------------
CSV_FILE = "expenses.csv"

def _initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(["Item", "Amount", "Category"])

def add_expense(item: str, amount: float, category: str) -> str:
    """Logs a new expense with item, amount, and category into CSV."""
    _initialize_csv()
    with open(CSV_FILE, mode='a', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([item, amount, category])
    return f"Successfully logged expense: {item} - ${amount} ({category})"

def get_expenses() -> str:
    """Retrieves all logged expenses from the CSV file."""
    _initialize_csv()
    with open(CSV_FILE, mode='r') as f:
        reader = csv.reader(f)
        rows = list(reader)
    if len(rows) <= 1:
        return "No expenses recorded yet."
    return "\n".join([", ".join(row) for row in rows])

# -------------------------------------------------------------
# 4. Wrap Tools as LangChain Structured Tools for LangGraph
# -------------------------------------------------------------
mcp_tools = [
    StructuredTool.from_function(
        func=add_expense,
        name="add_expense",
        description="Logs a new expense with item, amount, and category."
    ),
    StructuredTool.from_function(
        func=get_expenses,
        name="get_expenses",
        description="Retrieves all logged expenses from the expense tracker."
    )
]

# -------------------------------------------------------------
# 5. Build & Execute LangGraph Agent
# -------------------------------------------------------------
agent = create_react_agent(llm, mcp_tools)

async def run_agentic_workflow():
    print("--- Task 1: Log an expense ---")
    prompt_1 = "I bought a pizza for $12.50. Category is Food."
    response_1 = await agent.ainvoke({"messages": [("user", prompt_1)]})

    # Correct attribute inspection for LangChain BaseMessage objects
    for msg in response_1["messages"]:
        if msg.type == "ai" and msg.content:
            print(f"\n[Agent Response]: {msg.content}")

    print("\n--- Task 2: Retrieve records ---")
    prompt_2 = "Show me all expenses logged so far."
    response_2 = await agent.ainvoke({"messages": [("user", prompt_2)]})

    for msg in response_2["messages"]:
        if msg.type == "ai" and msg.content:
            print(f"\n[Agent Response]: {msg.content}")

# Run Execution Loop
asyncio.run(run_agentic_workflow())

/tmp/ipykernel_2360/3414576312.py:68: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, mcp_tools)


--- Task 1: Log an expense ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Response]: [{'type': 'text', 'text': "I've logged your expense: **pizza** for **$12.50** under the category **Food**.", 'extras': {'signature': 'EswBCskBARFNMg+572xMHaes/pWpa3DqTu0ZbWLRyQzhVF2KiiNXlKK4kxC0lBzp6gEWqrKqqrzWMZ9qCFrjpFDyYG/mKZvFdrcgARj4QCyC/3U8rK7RmSz85oDwf62w+rWvueIKgjvUUZqFDhowq/3Xj3OOCneCuOxBtR9E9fo3HuqhSUpOz+zch6JHL7kOLbsftqEoyd35ECTdYAy7pwQ4jhIqFDh6kxCueu/rYOo7QfqsQ0NtqnU8dIlUTzAkN9whwing3rIjSXaT/iPN'}}]

--- Task 2: Retrieve records ---


/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(
/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3908: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(



[Agent Response]: [{'type': 'text', 'text': 'Here are all the expenses logged so far:\n\n* **Item:** pizza\n* **Amount:** $12.50\n* **Category:** Food', 'extras': {'signature': 'Eu8BCuwBARFNMg9JKSLCWocYWH2umUjbNyG2PQyXd5xQnGX3weJp3CXxyCJ4oPwoAAI8d7/eFWBRuF/R1lbpvw78K+tqWXtwH+aWAc3bCHrKsLtgh2ZOZwiec4849vqZcYFs7qnCUXdPpPWR1VtnibGkuojeRwzJNMY0hCc6U0zyHxM6CGosCIIVIUDL6wiydeL8KRbvcuxR2t7W/VLJDQ80xBS8npCOuFqHZFzURw8xnS6ZUozsYnsV/CWiTnnnjSV13Se7PoK2m662ss2aD85iWWmnRYEjLJ7pobhxfRUttxNHYWPUPoZmViL/pHQZTDs='}}]


In [14]:
import os
import csv
import asyncio
import nest_asyncio
from google.colab import userdata

from langchain_core.tools import StructuredTool
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

nest_asyncio.apply()

# 1. API & Model Setup
os.environ["GOOGLE_API_KEY"] = userdata.get("Gemini_API_Key_2")
llm = ChatGoogleGenerativeAI(model="gemini-3.7-flash")

# 2. Expense Tools
CSV_FILE = "expenses.csv"

def _initialize_csv():
    if not os.path.exists(CSV_FILE):
        with open(CSV_FILE, mode='w', newline='') as f:
            csv.writer(f).writerow(["Item", "Amount", "Category"])

def add_expense(item: str, amount: float, category: str) -> str:
    _initialize_csv()
    with open(CSV_FILE, mode='a', newline='') as f:
        csv.writer(f).writerow([item, amount, category])
    return f"Logged: {item} - ${amount} ({category})"

def get_expenses() -> str:
    _initialize_csv()
    with open(CSV_FILE, mode='r') as f:
        rows = list(csv.reader(f))
    return "No expenses." if len(rows) <= 1 else "\n".join([", ".join(r) for r in rows[1:]])

# 3. Wrap Tools & Build Agent
mcp_tools = [
    StructuredTool.from_function(func=add_expense, name="add_expense", description="Logs an expense."),
    StructuredTool.from_function(func=get_expenses, name="get_expenses", description="Gets logged expenses.")
]

agent = create_agent(llm, mcp_tools)

# 4. Helper to extract raw text string
def get_clean_text(content) -> str:
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "".join([b.get("text", "") for b in content if isinstance(b, dict) and b.get("type") == "text"])
    return str(content)

# 5. Execution
async def run_agentic_workflow():
    res1 = await agent.ainvoke({"messages": [("user", "I bought a pizza for $12.50. Category is Food.")]})
    print(get_clean_text(res1["messages"][-1].content))

    res2 = await agent.ainvoke({"messages": [("user", "Show me all expenses logged so far.")]})
    print(get_clean_text(res2["messages"][-1].content))

asyncio.run(run_agentic_workflow())

Logged your expense: **pizza** for **$12.50** under the **Food** category.
Here are the expenses logged so far:

1. **Item:** Pizza | **Amount:** $12.50 | **Category:** Food
2. **Item:** Pizza | **Amount:** $12.50 | **Category:** Food
3. **Item:** Pizza | **Amount:** $12.50 | **Category:** Food

**Total:** $37.50
